In [ ]:
import os
import operator

from dotenv import load_dotenv
from typing import TypedDict, Annotated

from azure.identity import DefaultAzureCredential, get_bearer_token_provider

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    AnyMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

from langchain_community.tools.tavily_search import TavilySearchResults


# =========================================================
# Azure OpenAI
# =========================================================

load_dotenv()

openai_endpoint = os.getenv("ENDPOINT")
model_deployment = os.getenv("MODEL_DEPLOYMENT")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

llm = ChatOpenAI(
    model=model_deployment,
    base_url=openai_endpoint,
    api_key=token_provider(),
    temperature=0
)

# =========================================================
# Tool
# =========================================================

search_tool = TavilySearchResults(max_results=2, tavily_api_key="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")

# =========================================================
# Memory
# =========================================================

memory = InMemorySaver()

# =========================================================
# State
# =========================================================

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

# =========================================================
# Agent
# =========================================================

class Agent:

    def __init__(self, model, tools, checkpointer, system=""):

        self.system = system

        self.tools = {t.name: t for t in tools}

        self.model = model.bind_tools(tools)

        graph = StateGraph(AgentState)

        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)

        graph.add_conditional_edges(
            "llm",
            self.has_tool_call,
            {
                True: "action",
                False: END
            }
        )

        graph.add_edge("action", "llm")

        graph.set_entry_point("llm")

        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]   # Human approval point
        )

    def call_model(self, state):

        messages = state["messages"]

        if self.system:
            messages = [
                SystemMessage(content=self.system)
            ] + messages

        response = self.model.invoke(messages)

        return {
            "messages": [response]
        }

    def has_tool_call(self, state):

        last_message = state["messages"][-1]

        return (
            hasattr(last_message, "tool_calls")
            and len(last_message.tool_calls) > 0
        )

    def take_action(self, state):

        tool_calls = state["messages"][-1].tool_calls

        results = []

        for tc in tool_calls:

            print(f"\nExecuting Tool: {tc['name']}")

            result = self.tools[tc["name"]].invoke(
                tc["args"]
            )

            results.append(
                ToolMessage(
                    tool_call_id=tc["id"],
                    name=tc["name"],
                    content=str(result)
                )
            )

        return {
            "messages": results
        }

# =========================================================
# Prompt
# =========================================================

system_prompt = """
You are a research assistant.

Use search whenever needed.

Always explain your reasoning.
"""

# =========================================================
# Build Agent
# =========================================================

abot = Agent(
    llm,
    [search_tool],
    checkpointer=memory,
    system=system_prompt
)

# =========================================================
# Start Conversation
# =========================================================

thread = {
    "configurable": {
        "thread_id": "human-demo"
    }
}

for event in abot.graph.stream(
    {
        "messages": [
            HumanMessage(
                content="What is the weather in San Francisco?"
            )
        ]
    },
    thread
):
    print(event)

# =========================================================
# Show Interrupted State
# =========================================================

print("\n")
print("=" * 60)
print("CURRENT GRAPH STATE")
print("=" * 60)

state = abot.graph.get_state(thread)

print(state)

print("\nNEXT NODE:")
print(state.next)

# =========================================================
# Human Approval
# =========================================================

approval = input(
    "\nApprove tool execution? (y/n): "
)

if approval.lower() == "y":

    print("\nResuming graph...\n")

    for event in abot.graph.stream(
        None,
        thread
    ):
        print(event)

else:

    print("\nTool execution cancelled by user.")

C:\Users\a855099\AppData\Local\Temp\ipykernel_28764\2847069351.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
C:\Users\a855099\AppData\Local\Temp\ipykernel_28764\2847069351.py:48: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2, tavily_api_key="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")


{'llm': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 187, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 33, 'engine_ttlt_ms': 507, 'pre_inference_ms': 106, 'service_tbt_ms': 6, 'service_ttft_ms': 547, 'service_ttlt_ms': 1013, 'user_visible_ttft_ms': 440}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ESKCoekti4VG4kxoxGEJtZVxYPcgy', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0dd52-6abf-7fe1-8be6-3099bc8b7d60-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current weather San Franci